# Data Cleaning process

Here we are going to solve most common data clearing problems so you can get actually analyzing your data faster

### First Step
Always we have to start with a quick view of the data

In [19]:
import pandas as pd
import numpy as np

#read the data
nfl_data = pd.read_csv('/Users/caneda/Downloads/NFLDATA.csv')
nfl_data.head()

/var/folders/jf/_tzg95393slfqnz8lvks1m3m0000gn/T/ipykernel_60290/444427366.py:5: DtypeWarning: Columns (25,51) have mixed types. Specify dtype option on import or set low_memory=False.
  nfl_data = pd.read_csv('/Users/caneda/Downloads/NFLDATA.csv')


,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


### How many missing data points do we have?

In [20]:
missing_values_count = nfl_data.isna().sum()
print(missing_values_count)

Date             0
GameID           0
Drive            0
qtr              0
down         54218
             ...  
Win_Prob     21993
WPA           4817
airWPA      220738
yacWPA      220956
Season           0
Length: 102, dtype: int64


In [21]:
# how many total missing values do we have?
total_cells = np.prod(nfl_data.shape)
total_missing = missing_values_count.sum()
print('Total cells: %d' % total_cells)
print('Total missing values: %d' % total_missing)
# what percentage of the data is missing?
print('Percentage of data that is missing: %d%%' % (total_missing/total_cells*100))

Total cells: 36969594
Total missing values: 10222931
Percentage of data that is missing: 27%


More that a quart of data is empty. We have to go deeper inside data and figure out whats going on and how to fix it

### Ways to manage empty values
There are two posible cases: 
1. The data doesnt exist - We shouldnt try to guess data to fix empty values (leave empty cells with Nan value)
2. The data exist but It couldnt be recorded - We can apply some tecnichques to try to guess the missing values


In [22]:
missing_values_count[0:10]

Date                0
GameID              0
Drive               0
qtr                 0
down            54218
time              188
TimeUnder           0
TimeSecs          188
PlayTimeDiff      374
SideofField       450
dtype: int64

In [23]:
# remove all columns that contain a missing value
nfl_data_dropna = nfl_data.dropna(axis= 1)
print('Columns in original dataset: %d \n' % nfl_data.shape[1])
print('Columns in dataset after dropping columns with missing values: %d' % nfl_data_dropna.shape[1])

Columns in original dataset: 102 

Columns in dataset after dropping columns with missing values: 37


In [24]:
# replace all the Nan values with whatever value comes directly after the Nan in the same column
nfl_data_fillna = nfl_data.fillna(method='bfill', axis=0)
print('Columns in original dataset: %d \n' % nfl_data.shape[1])
print('Columns in dataset after replacing missing values: %d' % nfl_data_fillna.shape[1])
print(nfl_data_fillna.head())

#Cheching if there are any missing values left
missing_values_count_fillna = nfl_data_fillna.isna().sum()
print(missing_values_count_fillna)

/var/folders/jf/_tzg95393slfqnz8lvks1m3m0000gn/T/ipykernel_60290/2974249791.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nfl_data_fillna = nfl_data.fillna(method='bfill', axis=0)


Columns in original dataset: 102 

Columns in dataset after replacing missing values: 102
         Date      GameID  Drive  qtr  down   time  TimeUnder  TimeSecs  \
0  2009-09-10  2009091000      1    1   1.0  15:00         15    3600.0   
1  2009-09-10  2009091000      1    1   1.0  14:53         15    3593.0   
2  2009-09-10  2009091000      1    1   2.0  14:16         15    3556.0   
3  2009-09-10  2009091000      1    1   3.0  13:35         14    3515.0   
4  2009-09-10  2009091000      1    1   4.0  13:27         14    3507.0   

   PlayTimeDiff SideofField  ...    yacEPA  Home_WP_pre  Away_WP_pre  \
0           0.0         TEN  ...  1.146076     0.485675     0.514325   
1           7.0         PIT  ...  1.146076     0.546433     0.453567   
2          37.0         PIT  ... -5.031425     0.551088     0.448912   
3          41.0         PIT  ... -5.031425     0.510793     0.489207   
4           8.0         PIT  ...  0.163935     0.461217     0.538783   

   Home_WP_post  Away_WP_p

## Intermidate level, handling with missing value
Most machine learning libraries (including scikit-learn) would give an error while trying to build a model with empty data cells. So will need to choose an estrategy to fill this empty data cells

### 1) Simple Option: Drop Columsn with missing Values
### 2) Imputation: Fill the missing value with some number
### 3) An Extension To Imputation: Add extra column to indicate if values were missing or not
    Option 3 - Will be helpfull for some models but sometimes it will not have impact to the model accuracy

In [25]:
from sklearn.model_selection import train_test_split

data_homes = pd.read_csv('/Users/caneda/Downloads/home-data-for-ml-course/train.csv')

#Select the target and predictor variables
y = data_homes.SalePrice
melb_predictors = data_homes.drop(['SalePrice'], axis=1)
X = melb_predictors.select_dtypes(exclude=['object'])

#Diveide data into training and validation data, for both predictors and target
train_X, val_X, train_y, val_y = train_test_split(X, y, train_size= 0.8, test_size= 0.2, random_state= 0)

### Define function to measure Quality of Each Approach
Compare different approaches to dealing with missing values. This function reports the mean absolute error (MAE) from random forest model

In [26]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

#Function for comparing different approaches to handling missing values
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [28]:
#Score from Approach 1 (drop columns with missing values)
cols_with_missing = [col for col in train_X.columns if train_X[col].isnull().any()]
reduced_train_X = train_X.drop(cols_with_missing, axis=1)
reduced_val_X = val_X.drop(cols_with_missing, axis=1)
print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_train_X, reduced_val_X, train_y, val_y))
#Score from Approach 2 (imputation)
from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer()
imputed_train_X = pd.DataFrame(my_imputer.fit_transform(train_X))
imputed_val_X = pd.DataFrame(my_imputer.transform(val_X))

#Imputation removed column names; put them back
imputed_train_X.columns = train_X.columns
imputed_val_X.columns = val_X.columns
print("MAE from Approach 2 (Imputation):")
print(score_dataset(imputed_train_X, imputed_val_X, train_y, val_y))

#Score from Approach 3 (imputation with indicator for missingness)
X_train_plus = train_X.copy()
X_valid_plus = val_X.copy()

#Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

#Imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

#Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns
print("MAE from Approach 3 (Imputation with indicator for missingness):")
print(score_dataset(imputed_X_train_plus, imputed_X_valid_plus, train_y, val_y))

MAE from Approach 1 (Drop columns with missing values):
17952.591404109586
MAE from Approach 2 (Imputation):
18250.608013698627
MAE from Approach 3 (Imputation with indicator for missingness):
18253.31479452055


##### As a result we can see that dropping columns (Approach 1) will give us the best result